In [1]:
# -*- coding: utf-8 -*-
# @Time    : 2020/4/4 8:18
# @Author  : Hui Wang

from collections import defaultdict
import random
import numpy as np
import pandas as pd
import json
import pickle
import gzip
import tqdm
import os
from tqdm import tqdm

true=True
false=False
def parse(path): # for Amazon
    g = gzip.open(path, 'rb')
    inter_list = []
    for l in tqdm(g):
        inter_list.append(json.loads(l.decode()))
        # inter_list.append(eval(l))

    return inter_list

def parse_meta_2023(path): # for Amazon
    g = gzip.open(path, 'rb')
    inter_list = {}
    for l in tqdm(g):
        info = json.loads(l.decode())
        inter_list[info['parent_asin']] = info
    return inter_list
    

def parse_meta(path): # for Amazon
    g = gzip.open(path, 'rb')
    inter_list = {}

    for l in tqdm(g):
        info = eval(l)

        if not isinstance(info, dict):
            
            continue

        inter_list[info['asin']] = info
    return inter_list


# return (user item timestamp) sort in get_interaction
def Amazon2023(dataset_name, rating_score):

    datas = []
    # older Amazon
    data_flie = './data/' + str(dataset_name) + '/raw/' + str(dataset_name) + '.jsonl.gz'
    # latest Amazon
    # data_flie = '/home/hui_wang/data/new_Amazon/' + dataset_name + '.json.gz'
    deletedCnt = 0
    for inter in parse(data_flie):
        
        if float(inter['rating']) <= rating_score: # 小于一定分数去掉
            deletedCnt += 1
            print(f"inter rating: {inter['rating']} too low, deleted {deletedCnt} inter {inter}" )
            
            continue
        user = inter['user_id']
        item = inter['asin']
        time = inter['timestamp']
        datas.append((user, item, int(time)))
    return datas

# return (user item timestamp) sort in get_interaction
def Amazon(dataset_name, rating_score):
    '''
    reviewerID - ID of the reviewer, e.g. A2SUAM1J3GNN3B
    asin - ID of the product, e.g. 0000013714
    reviewerName - name of the reviewer
    helpful - helpfulness rating of the review, e.g. 2/3
    --"helpful": [2, 3],
    reviewText - text of the review
    --"reviewText": "I bought this for my husband who plays the piano. ..."
    overall - rating of the product
    --"overall": 5.0,
    summary - summary of the review
    --"summary": "Heavenly Highway Hymns",
    unixReviewTime - time of the review (unix time)
    --"unixReviewTime": 1252800000,
    reviewTime - time of the review (raw)
    --"reviewTime": "09 13, 2009"
    '''
    datas = []
    # older Amazon
    data_flie = './data/' + str(dataset_name) + '/raw/' + str(dataset_name) + '_2018.json.gz'
    # latest Amazon
    # data_flie = '/home/hui_wang/data/new_Amazon/' + dataset_name + '.json.gz'
    for inter in parse(data_flie):
        if float(inter['overall']) <= rating_score: # 小于一定分数去掉
            continue
        user = inter['reviewerID']
        item = inter['asin']
        time = inter['unixReviewTime']
        datas.append((user, item, int(time)))
    return datas

def New_Amazon(dataset_name, rating_score):
    '''
    reviewerID - ID of the reviewer, e.g. A2SUAM1J3GNN3B
    asin - ID of the product, e.g. 0000013714
    reviewerName - name of the reviewer
    helpful - helpfulness rating of the review, e.g. 2/3
    --"helpful": [2, 3],
    reviewText - text of the review
    --"reviewText": "I bought this for my husband who plays the piano. ..."
    overall - rating of the product
    --"overall": 5.0,
    summary - summary of the review
    --"summary": "Heavenly Highway Hymns",
    unixReviewTime - time of the review (unix time)
    --"unixReviewTime": 1252800000,
    reviewTime - time of the review (raw)
    --"reviewTime": "09 13, 2009"
    '''
    datas = []
    # older Amazon
    data_flie = './data/' + str(dataset_name) + '/raw/' + str(dataset_name) + '.json.gz'
    # latest Amazon
    # data_flie = '/home/hui_wang/data/new_Amazon/' + dataset_name + '.json.gz'
    for inter in parse(data_flie):
        if float(inter['overall']) <= rating_score: # 小于一定分数去掉
            continue
        user = inter['reviewerID']
        item = inter['asin']
        time = inter['unixReviewTime']
        datas.append((user, item, int(time)))
    return datas


def Amazon_meta2023(dataset_name, item_set):
    ''' fixme:
    asin - ID of the product, e.g. 0000031852
    --"asin": "0000031852",
    title - name of the product
    --"title": "Girls Ballet Tutu Zebra Hot Pink",
    description
    price - price in US dollars (at time of crawl)
    --"price": 3.17,
    imUrl - url of the product image (str)
    --"imUrl": "http://ecx.images-amazon.com/images/I/51fAmVkTbyL._SY300_.jpg",
    related - related products (also bought, also viewed, bought together, buy after viewing)
    --"related":{
        "also_bought": ["B00JHONN1S"],
        "also_viewed": ["B002BZX8Z6"],
        "bought_together": ["B002BZX8Z6"]
    },
    salesRank - sales rank information
    --"salesRank": {"Toys & Games": 211836}
    brand - brand name
    --"brand": "Coxlures",
    categories - list of categories the product belongs to
    --"categories": [["Sports & Outdoors", "Other Sports", "Dance"]]
    '''
    datas = {}
    meta_flie = './data/' + str(dataset_name) + '/raw/meta_' + str(dataset_name) + '.jsonl.gz'

    meta_list = parse_meta_2023(meta_flie)
    no_meta_item = []
    for asin in tqdm(item_set):
        if asin not in meta_list.keys():
            no_meta_item.append(asin)
            continue
        datas[asin] = meta_list[asin]
    print(f'no meta item: {len(no_meta_item)}')
    
    return datas, no_meta_item

def Amazon_meta(dataset_name, item_set):
    '''
    asin - ID of the product, e.g. 0000031852
    --"asin": "0000031852",
    title - name of the product
    --"title": "Girls Ballet Tutu Zebra Hot Pink",
    description
    price - price in US dollars (at time of crawl)
    --"price": 3.17,
    imUrl - url of the product image (str)
    --"imUrl": "http://ecx.images-amazon.com/images/I/51fAmVkTbyL._SY300_.jpg",
    related - related products (also bought, also viewed, bought together, buy after viewing)
    --"related":{
        "also_bought": ["B00JHONN1S"],
        "also_viewed": ["B002BZX8Z6"],
        "bought_together": ["B002BZX8Z6"]
    },
    salesRank - sales rank information
    --"salesRank": {"Toys & Games": 211836}
    brand - brand name
    --"brand": "Coxlures",
    categories - list of categories the product belongs to
    --"categories": [["Sports & Outdoors", "Other Sports", "Dance"]]
    '''
    datas = {}
    meta_flie = './data/' + str(dataset_name) + '/raw/meta_' + str(dataset_name) + '_2018.json.gz'
    meta_list = parse_meta(meta_flie)
    no_meta_item = []
    for asin in tqdm(item_set):
        if asin not in meta_list.keys():
            no_meta_item.append(asin)
            continue
        datas[asin] = meta_list[asin]
    return datas, no_meta_item

def Yelp(date_min, date_max, rating_score): # take out inters in [date_min, date_max] and the score < rating_score
    datas = []
    data_flie = './data/yelp/raw/yelp_academic_dataset_review.json'
    lines = open(data_flie).readlines()
    for line in tqdm(lines):
        review = json.loads(line.strip())
        user = review['user_id']
        item = review['business_id']
        rating = review['stars']
        # 2004-10-12 10:13:32 2019-12-13 15:51:19
        date = review['date']
        # 剔除一些例子
        if date < date_min or date > date_max or float(rating) <= rating_score:
            continue
        time = date.replace('-','').replace(':','').replace(' ','') 
        datas.append((user, item, int(time)))
    return datas


def Yelp_meta(datamaps):
    meta_infos = {}
    meta_file = './data/yelp/raw/yelp_academic_dataset_business.json'
    item_ids = list(datamaps['item2id'].keys())
    lines = open(meta_file).readlines()
    for line in tqdm(lines):
        info = json.loads(line)
        if info['business_id'] not in item_ids:
            continue
        meta_infos[info['business_id']] = info
    return meta_infos


def add_comma(num):
    # 1000000 -> 1,000,000
    str_num = str(num)
    res_num = ''
    for i in range(len(str_num)):
        res_num += str_num[i]
        if (len(str_num)-i-1) % 3 == 0:
            res_num += ','
    return res_num[:-1]

# categories 和 brand is all attribute
def get_attribute_Amazon(meta_infos, datamaps, attribute_core):

    attributes = defaultdict(int)
    # for iid, info in tqdm.tqdm(meta_infos.items()):
    #     for cates in info['categories']:
    #         for cate in cates[1:]: # 把主类删除 没有用
    #             attributes[cate] +=1
    #     try:
    #         attributes[info['brand']] += 1
    #     except:
    #         pass

    # print(f'before delete, attribute num:{len(attributes)}')
    # new_meta = {}
    # for iid, info in tqdm.tqdm(meta_infos.items()):
    #     new_meta[iid] = []

    #     try:
    #         if attributes[info['brand']] >= attribute_core:
    #             new_meta[iid].append(info['brand'])
    #     except:
    #         pass
    #     for cates in info['categories']:
    #         for cate in cates[1:]:
    #             if attributes[cate] >= attribute_core:
    #                 new_meta[iid].append(cate)
    # 做映射
    attribute2id = {}
    id2attribute = {}
    attributeid2num = defaultdict(int)
    attribute_id = 1
    items2attributes = {}
    attribute_lens = []

    for iid, attributes in meta_infos.items():
        item_id = datamaps['item2id'][iid]
        items2attributes[item_id] = []
        for attribute in attributes:
            if attribute not in attribute2id:
                attribute2id[attribute] = attribute_id
                id2attribute[attribute_id] = attribute
                attribute_id += 1
            attributeid2num[attribute2id[attribute]] += 1
            items2attributes[item_id].append(attribute2id[attribute])
        attribute_lens.append(len(items2attributes[item_id]))
    print(f'before delete, attribute num:{len(attribute2id)}')
    print(f'attributes len, Min:{np.min(attribute_lens)}, Max:{np.max(attribute_lens)}, Avg.:{np.mean(attribute_lens):.4f}')
    # 更新datamap
    datamaps['attribute2id'] = attribute2id
    datamaps['id2attribute'] = id2attribute
    datamaps['attributeid2num'] = attributeid2num
    return len(attribute2id), np.mean(attribute_lens), datamaps, items2attributes


def get_attribute_Yelp(meta_infos, datamaps, attribute_core):
    attributes = defaultdict(int)
    for iid, info in tqdm(meta_infos.items()):
        try:
            cates = [cate.strip() for cate in info['categories'].split(',')]
            for cate in cates:
                attributes[cate] +=1
        except:
            pass
    print(f'before delete, attribute num:{len(attributes)}')
    new_meta = {}
    for iid, info in tqdm(meta_infos.items()):
        new_meta[iid] = []
        try:
            cates = [cate.strip() for cate in info['categories'].split(',') ]
            for cate in cates:
                if attributes[cate] >= attribute_core:
                    new_meta[iid].append(cate)
        except:
            pass
    # 做映射
    attribute2id = {}
    id2attribute = {}
    attribute_id = 1
    items2attributes = {}
    attribute_lens = []
    # load id map
    for iid, attributes in new_meta.items():
        item_id = datamaps['item2id'][iid]
        items2attributes[item_id] = []
        for attribute in attributes:
            if attribute not in attribute2id:
                attribute2id[attribute] = attribute_id
                id2attribute[attribute_id] = attribute
                attribute_id += 1
            items2attributes[item_id].append(attribute2id[attribute])
        attribute_lens.append(len(items2attributes[item_id]))
    print(f'after delete, attribute num:{len(attribute2id)}')
    print(f'attributes len, Min:{np.min(attribute_lens)}, Max:{np.max(attribute_lens)}, Avg.:{np.mean(attribute_lens):.4f}')
    # 更新datamap
    datamaps['attribute2id'] = attribute2id
    datamaps['id2attribute'] = id2attribute
    return len(attribute2id), np.mean(attribute_lens), datamaps, items2attributes

def get_interaction(datas): # sort the interactions based on timestamp
    user_seq = {}
    
    
    for data in datas:
        user, item, time = data
        if user in user_seq:             
            if item not in [i for i, _ in user_seq[user]]:
                user_seq[user].append((item, time))
            # user_seq[user].append((item, time))
        else:
            user_seq[user] = []
            user_seq[user].append((item, time))

    for user, item_time in user_seq.items():
        item_time.sort(key=lambda x: x[1])  # 对各个数据集得单独排序
        items = []
        for t in item_time:
            items.append(t[0])
        user_seq[user] = items
        
    return user_seq

def filter_core(user_seq, user_core, item_core):
    user_count = defaultdict(int)
    item_count = defaultdict(int)
    for user, items in user_seq.items():
        user_count[user] = len(items)
        for item in items:
            item_count[item] += 1
    
    filtered_user_seq = {}
    for user, items in user_seq.items():
        new_items = []
        for item in items:
            if item_count[item] >= item_core:
                new_items.append(item)
        if len(new_items) >= user_core:
            filtered_user_seq[user] = new_items

    return filtered_user_seq     
            

# K-core user_core item_core
def check_Kcore(user_items, user_core, item_core):
    user_count = defaultdict(int)
    item_count = defaultdict(int)
    for user, items in user_items.items():
        for item in items:
            user_count[user] += 1
            item_count[item] += 1

    for user, num in user_count.items():
        if num < user_core:
            return user_count, item_count, False
    for item, num in item_count.items():
        if num < item_core:
            return user_count, item_count, False
    return user_count, item_count, True # 已经保证Kcore

# 循环过滤 K-core
def filter_Kcore(user_items, user_core, item_core): # user 接所有items
    user_count, item_count, isKcore = check_Kcore(user_items, user_core, item_core)
    while not isKcore:
        for user, num in user_count.items():
            if user_count[user] < user_core: # 直接把user 删除
                user_items.pop(user)
            else:
                for item in user_items[user]:
                    if item_count[item] < item_core:
                        user_items[user].remove(item)
        user_count, item_count, isKcore = check_Kcore(user_items, user_core, item_core)
    return user_items


def filter_common(user_items, user_t, item_t):

    user_count = defaultdict(int)
    item_count = defaultdict(int)
    for user, item, _ in user_items:
        user_count[user] += 1
        item_count[item] += 1

    User = {}
    for user, item, timestamp in user_items:
        if user_count[user] < user_t or item_count[item] < item_t:
            continue
        if user not in User.keys():
            User[user] = []
        if item not in [i for i, _ in User[user]]:
            User[user].append((item, timestamp))

    new_User = {}
    for userid in User.keys():
        User[userid].sort(key=lambda x: x[1])
        new_hist = [i for i, t in User[userid]]
        new_User[userid] = new_hist

    return new_User



def id_map(user_items): # user_items dict

    user2id = {} # raw 2 uid
    item2id = {} # raw 2 iid
    id2user = {} # uid 2 raw
    id2item = {} # iid 2 raw
    user_id = 1
    item_id = 1
    final_data = {}
    for user, items in user_items.items():
        if user not in user2id:
            user2id[user] = str(user_id)
            id2user[str(user_id)] = user
            user_id += 1
        iids = [] # item id lists
        for item in items:
            if item not in item2id:
                item2id[item] = str(item_id)
                id2item[str(item_id)] = item
                item_id += 1
            iids.append(item2id[item])
        uid = user2id[user]
        final_data[uid] = iids
    data_maps = {
        'user2id': user2id,
        'item2id': item2id,
        'id2user': id2user,
        'id2item': id2item
    }
    return final_data, user_id-1, item_id-1, data_maps


def get_counts(user_items):

    user_count = {}
    item_count = {}

    for user, items in user_items.items():
        user_count[user] = len(items)
        for item in items:
            if item not in item_count.keys():
                item_count[item] = 1
            else:
                item_count[item] += 1
    print(f'User Num: {len(user_count)}, Item Num: {len(item_count)} user_items len: {len(user_items)}')
    return user_count, item_count


def filter_minmum(user_items, min_len=3):

    new_user_items = {}
    for user, items in user_items.items():
        if len(items) >= min_len:
            new_user_items[user] = items

    return new_user_items

def remove_items(user_items, missing_items):
    new_user_items = {}
    for user, items in tqdm(user_items.items()):
        new_items = []
        for item in items:
            if item not in missing_items:
                new_items.append(item)
        if len(new_items) > 0:
            new_user_items[user] = new_items
    return new_user_items

In [2]:


def LastFM():
    user_core = 5
    item_core = 5
    datas = []
    data_file = '/path/lastfm/2k/user_attributegedartists-timestamps.dat'
    lines = open(data_file).readlines()
    for line in tqdm.tqdm(lines[1:]):
        user, item, attribute, timestamp = line.strip().split('\t')
        datas.append((user, item, int(timestamp)))

    # 有重复item
    user_seq = {}
    user_seq_notime = {}
    for data in datas:
        user, item, time = data
        if user in user_seq:
            if item not in user_seq_notime[user]:
                user_seq[user].append((item, time))
                user_seq_notime[user].append(item)
            else:
                continue
        else:
            user_seq[user] = []
            user_seq_notime[user] = []

            user_seq[user].append((item, time))
            user_seq_notime[user].append(item)

    for user, item_time in user_seq.items():
        item_time.sort(key=lambda x: x[1])  # 对各个数据集得单独排序
        items = []
        for t in item_time:
            items.append(t[0])
        user_seq[user] = items

    user_items = filter_Kcore(user_seq, user_core=user_core, item_core=item_core)
    print(f'User {user_core}-core complete! Item {item_core}-core complete!')

    user_items, user_num, item_num, data_maps = id_map(user_items)  # new_num_id
    user_count, item_count, _ = check_Kcore(user_items, user_core=user_core, item_core=item_core)
    user_count_list = list(user_count.values())
    user_avg, user_min, user_max = np.mean(user_count_list), np.min(user_count_list), np.max(user_count_list)
    item_count_list = list(item_count.values())
    item_avg, item_min, item_max = np.mean(item_count_list), np.min(item_count_list), np.max(item_count_list)
    interact_num = np.sum([x for x in user_count_list])
    sparsity = (1 - interact_num / (user_num * item_num)) * 100
    show_info = f'Total User: {user_num}, Avg User: {user_avg:.4f}, Min Len: {user_min}, Max Len: {user_max}\n' + \
                f'Total Item: {item_num}, Avg Item: {item_avg:.4f}, Min Inter: {item_min}, Max Inter: {item_max}\n' + \
                f'Iteraction Num: {interact_num}, Sparsity: {sparsity:.2f}%'
    print(show_info)

    attribute_file = './data_path/artist2attributes.json'

    meta_item2attribute = json.loads(open(attribute_file).readline())

    # 做映射
    attribute2id = {}
    id2attribute = {}
    attribute_id = 1
    item2attributes = {}
    attribute_lens = []
    # load id map
    for iid, attributes in meta_item2attribute.items():
        if iid in list(data_maps['item2id'].keys()):
            item_id = data_maps['item2id'][iid]
            item2attributes[item_id] = []
            for attribute in attributes:
                if attribute not in attribute2id:
                    attribute2id[attribute] = attribute_id
                    id2attribute[attribute_id] = attribute
                    attribute_id += 1
                item2attributes[item_id].append(attribute2id[attribute])
            attribute_lens.append(len(item2attributes[item_id]))
    print(f'after delete, attribute num:{len(attribute2id)}')
    print(f'attributes len, Min:{np.min(attribute_lens)}, Max:{np.max(attribute_lens)}, Avg.:{np.mean(attribute_lens):.4f}')
    # 更新datamap
    data_maps['attribute2id'] = attribute2id
    data_maps['id2attribute'] = id2attribute

    data_name = 'LastFM'
    print(f'{data_name} & {add_comma(user_num)}& {add_comma(item_num)} & {user_avg:.1f}'
          f'& {item_avg:.1f}& {add_comma(interact_num)}& {sparsity:.2f}\%&{add_comma(len(attribute2id))}&'
          f'{np.mean(attribute_lens):.1f} \\')

    # -------------- Save Data ---------------
    # one user one line
    data_file = 'data/' + data_name + '.txt'
    item2attributes_file = 'data/' + data_name + '_item2attributes.json'

    with open(data_file, 'w') as out:
        for user, items in user_items.items():
            out.write(user + ' ' + ' '.join(items) + '\n')

    json_str = json.dumps(item2attributes)
    with open(item2attributes_file, 'w') as out:
        out.write(json_str)

# amazon_datas = ['Beauty', 'Sports_and_Outdoors', 'Toys_and_Games']


In [3]:
# def main(data_name, data_type='Amazon', user_core=3, item_core=3):
data_name = "clothing"
data_type = "Amazon"
user_core=5
item_core=5

assert data_type in {'Amazon', 'Yelp', 'Amazon_2023'}
np.random.seed(12345)
rating_score = 0.0  # rating score smaller than this score would be deleted
# user 5-core item 5-core
attribute_core = 0

if data_type == 'Yelp':
    date_max = '2019-12-31 00:00:00'
    date_min = '2000-01-01 00:00:00'
    datas = Yelp(date_min, date_max, rating_score)
elif data_type == "Amazon_2023":
    datas = Amazon2023(data_name, rating_score=rating_score)
else:
    datas = Amazon(data_name, rating_score=rating_score)


11285464it [01:55, 97921.16it/s] 


In [8]:
user_items = filter_common(datas, user_t=user_core, item_t=item_core)
print(len(user_items))
user_items = filter_minmum(user_items, min_len=5)


1219594


In [ ]:


print(f"debugging user_items len: {len(user_items)}")

print(f'{data_name} Raw data has been processed! Lower than {rating_score} are deleted!')

item_set = set()
for user, items in user_items.items():
    for item in items:
        item_set.add(item)
print(f"len of item_set: {len(item_set)}")
if data_type == 'Amazon':
    meta_infos, missing_items = Amazon_meta(data_name, item_set)
elif data_type == "Amazon_2023":
    meta_infos, missing_items = Amazon_meta2023(data_name, item_set)
    print("meta_infos len: ", len(meta_infos))

else:
    raise ValueError("data type not supported")

print(f"missing_items len: {len(missing_items)}")
print(f"meta_infos len: {len(meta_infos)}")
user_items = remove_items(user_items, missing_items)

print(f"user_items len: {len(user_items)} after parse meta_infos")
user_items, user_num, item_num, data_maps = id_map(user_items)  # new_num_id


print(f"debugging user_items len: {len(user_items)}")
# user_count, item_count, _ = check_Kcore(user_items, user_core=user_core, item_core=item_core)
user_count, item_count = get_counts(user_items)
user_count_list = list(user_count.values())
user_avg, user_min, user_max = np.mean(user_count_list), np.min(user_count_list), np.max(user_count_list)
item_count_list = list(item_count.values())
item_avg, item_min, item_max = np.mean(item_count_list), np.min(item_count_list), np.max(item_count_list)
interact_num = np.sum([x for x in user_count_list])
sparsity = (1 - interact_num / (user_num * item_num)) * 100
show_info = f'Total User: {user_num}, Avg User: {user_avg:.4f}, Min Len: {user_min}, Max Len: {user_max}\n' + \
            f'Total Item: {item_num}, Avg Item: {item_avg:.4f}, Min Inter: {item_min}, Max Inter: {item_max}\n' + \
            f'Iteraction Num: {interact_num}, Sparsity: {sparsity:.2f}%'
print(show_info)




debugging user_items len: 1165240
clothing Raw data has been processed! Lower than 0.0 are deleted!
len of item_set: 376432


2685059it [05:34, 8020.41it/s] 
100%|██████████| 376432/376432 [00:00<00:00, 417999.51it/s]


missing_items len: 0
meta_infos len: 376432


100%|██████████| 1165240/1165240 [00:12<00:00, 91792.27it/s] 


user_items len: 1165240 after parse meta_infos


KeyError: 'B002DD2Q0Y'

debugging user_items len: 1165240
User Num: 1165240, Item Num: 376432 user_items len: 1165240
Total User: 1165240, Avg User: 9.1855, Min Len: 5, Max Len: 491
Total Item: 376432, Avg Item: 28.4336, Min Inter: 1, Max Inter: 10297
Iteraction Num: 10703317, Sparsity: 100.00%


In [11]:


handled_path = 'data/' + data_name + '/handled/'
if not os.path.exists(handled_path):
    os.makedirs(handled_path)

data_file = handled_path + 'inter_seq.txt'
with open(data_file, 'w') as out:
    for user, items in user_items.items():
        out.write(user + ' ' + ' '.join(items) + '\n')
data = {} 
with open(f"data/{data_name}/handled/inter_seq.txt", 'r') as f:
    for line in tqdm(f):
        line_data = line.rstrip().split(' ')
        user_id = line_data[0]
        line_data.pop(0)    # delete user_id
        data[user_id] = line_data
with open(f"data/{data_name}/handled/inter.txt", 'w') as f:
    for user, item_list in tqdm(data.items()):
        for item in item_list:
            u = int(user)
            i = int(item)
            f.write('%s %s\n' % (u, i))



print(f'{data_name} & {add_comma(user_num)}& {add_comma(item_num)} & {user_avg:.1f}'
        f'& {item_avg:.1f}& {add_comma(interact_num)}& {sparsity:.2f} \\')

# # -------------- Save Data ---------------

item2attributes_file = handled_path + 'item2attributes.json'
id_file = handled_path + "id_map.json"


json_str = json.dumps(meta_infos)
with open(item2attributes_file, 'w') as out:
    out.write(json_str)
with open(id_file, "w") as f:
    json.dump(data_maps, f)



1165240it [00:07, 151379.33it/s]
100%|██████████| 1165240/1165240 [00:06<00:00, 193883.37it/s]


clothing & 1,165,240& 376,432 & 9.2& 28.4& 10,703,317& 100.00 \


In [ ]:
# for name in amazon_datas:
#     main(name, data_type='Amazon')
# main('beauty', data_type='Amazon_2023', user_core=5, item_core=5)

# main("electronics", data_type="Amazon")
# main("game", data_type="Amazon")
# main("music", data_type="Amazon")
# main("beauty", data_type="Amazon", user_core=5, item_core=5)
# LastFM()